### Problem 001: Kth Largest Element in a Stream (LeetCode 703)

### Problem Definition and Constraints
Design a class to find the $k$-th largest integer in a stream of values, including duplicates. The stream is not necessarily sorted.
Implement the `KthLargest` class:
* `KthLargest(int k, int[] nums)` Initializes the object with the integer `k` and the stream of integers `nums`.
* `int add(int val)` Appends the integer `val` to the stream and returns the $k$-th largest element in the stream.

* Constraints:
  * 1 <= k <= 10^4
  * 0 <= nums.length <= 10^4
  * -10^4 <= nums[i], val <= 10^4
  * There will always be at least `k` integers in the stream when you search for the $k$-th integer.

### Brute Force Approach
Every time a new number is added via `add(val)`, we append it to a standard array and then sort the entire array in descending order. Once sorted, we simply return the element at index `k - 1`.
* Time Complexity: $O(m \cdot n \log n)$ — Where $m$ is the number of `add` calls and $n$ is the total elements. Sorting the entire list on every single insertion is extremely slow.
* Space Complexity: $O(n)$ — We store every single number that ever gets added to the stream.

### Optimized Approach (Min-Heap)
To achieve $O(\log k)$ insertions, we use a Min-Heap. By strictly limiting the size of the heap to exactly `k` elements, the heap will naturally hold only the `k` largest numbers seen so far. Because it is a *Min*-Heap, the smallest number out of those `k` largest numbers will always sit at the very top (index 0). When `add()` is called, we push the new value onto the heap. If the heap's size exceeds `k`, we immediately pop the top element (which removes the smallest number, kicking out the "poorest" VIP). We then return the new top element.
* Time Complexity: $O(n \log k)$ for initialization, and $O(\log k)$ for each `add()` call. Pushing and popping from a heap of size $k$ takes logarithmic time relative to $k$.
* Space Complexity: $O(k)$ — We permanently discard any numbers that aren't in the top `k`, so our memory footprint never grows beyond size `k`.

In [ ]:
import heapq
from typing import List

class KthLargest:

    def __init__(self, k: int, nums: List[int]):
        # Store k so we know the maximum size of our VIP club
        self.k = k
        self.minHeap = nums
        
        # heapq.heapify transforms a standard list into a Min-Heap in-place in O(n) time
        heapq.heapify(self.minHeap)
        
        # If the starting array has more than k elements, pop the smallest ones 
        # until we are down to exactly k elements.
        while len(self.minHeap) > self.k:
            heapq.heappop(self.minHeap)

    def add(self, val: int) -> int:
        # 1. Push the new person into the club
        heapq.heappush(self.minHeap, val)
        
        # 2. If the club has more than k people, kick out the poorest one
        if len(self.minHeap) > self.k:
            heapq.heappop(self.minHeap)
            
        # 3. The poorest person in the VIP club (the root of the min-heap) 
        # is the k-th largest element overall.
        return self.minHeap[0]

# Your KthLargest object will be instantiated and called as such:
# obj = KthLargest(k, nums)
# param_1 = obj.add(val)


### Problem 002: Last Stone Weight (LeetCode 1046)

### Problem Definition and Constraints
You are given an array of integers `stones` where `stones[i]` represents the weight of the $i$-th stone.
We repeatedly choose the two heaviest stones and smash them together. 
* If `x == y`, both stones are destroyed.
* If `x < y`, the stone of weight `x` is destroyed, and the stone of weight `y` has a new weight of `y - x`.
Return the weight of the last remaining stone or return 0 if none remain.

* Constraints:
  * 1 <= stones.length <= 30
  * 1 <= stones[i] <= 1000

### Examples
* **Example 1:**
  * Input: `stones = [2,7,4,1,8,1]`
  * Output: `1`
  * Explanation: 
    * Smash 8 and 7 -> 1. Array becomes `[2,4,1,1,1]`
    * Smash 4 and 2 -> 2. Array becomes `[2,1,1,1]`
    * Smash 2 and 1 -> 1. Array becomes `[1,1,1]`
    * Smash 1 and 1 -> 0. Array becomes `[1]`
    * Last stone is 1.

### Brute Force Approach
The naive approach is to use a standard array. Inside a `while` loop, we sort the entire array in descending order, pop the first two elements, calculate their difference, and append the result back to the array. We repeat this until the array length is 1 or 0.
* Time Complexity: $O(n^2 \log n)$ — We have to re-sort the entire array of size $n$ every single time we do a smash operation (which happens roughly $n$ times).
* Space Complexity: $O(1)$ or $O(n)$ depending on if the sorting is done in-place.

### Optimized Approach (Max-Heap Simulation)
To avoid resorting the whole array every time, we use a Max-Heap. Because Python's `heapq` library only supports Min-Heaps, we first negate all values in the array (e.g., `5` becomes `-5`). The "heaviest" stone becomes the "smallest" negative number, naturally bubbling to the top of the Min-Heap.
While there is more than 1 stone in the heap, we pop the top two elements (multiplying by -1 to get their true positive weights back). If the first is heavier than the second, we calculate `first - second`, negate it, and push it back into the heap. If they are equal, we push nothing. We return the final stone (made positive again) or `0` if the heap is empty.
* Time Complexity: $O(n \log n)$ — `heapify` takes $O(n)$. We then do at most $n$ smashes. Each smash requires two `heappop` and one `heappush` operations, taking $O(\log n)$ time each.
* Space Complexity: $O(n)$ — We store the negated weights in a heap structure of size $n$.


In [ ]:
import heapq
from typing import List

class Solution:
    def lastStoneWeight(self, stones: List[int]) -> int:
        
        # 1. Transform into a Max-Heap using the "Negative Trick"
        # We multiply every stone by -1 so the heaviest stones become the smallest numbers
        max_heap = [-s for s in stones]
        heapq.heapify(max_heap)
        
        # 2. Simulate the gladiator arena
        # We need at least 2 stones to have a fight
        while len(max_heap) > 1:
            
            # Pop the two "smallest" (most negative) numbers and make them positive again
            first = -heapq.heappop(max_heap)   # The absolute heaviest stone
            second = -heapq.heappop(max_heap)  # The second heaviest stone
            
            # If the first is bigger, there is a leftover piece.
            if first > second:
                leftover = first - second
                # Push the leftover back into the ring (remember to make it negative!)
                heapq.heappush(max_heap, -leftover)
                
            # If first == second, both are destroyed. We do nothing and loop again.
            
        # 3. Check the aftermath
        # If there is a survivor, make it positive and return it.
        if max_heap:
            return -max_heap[0]
            
        # If they completely wiped each other out, return 0.
        return 0

### Problem 003: K Closest Points to Origin (LeetCode 973)

### Problem Definition and Constraints
You are given an array of coordinates `points` and an integer `k`. You need to return the `k` points that are closest to the origin (0, 0).
* The distance is measured using the standard Euclidean formula: $\sqrt{x^2 + y^2}$
* You can return the answer in any order.

* Constraints:
  * 1 <= k <= points.length <= 1000
  * -100 <= points[i][0], points[i][1] <= 100

### Brute Force Approach
Calculate the exact distance for every single point. Save them all in an array, and then fully sort the array from smallest distance to largest distance. Finally, slice off the first `k` elements.
* **Time Complexity:** $O(N \log N)$ — Sorting the entire list of $N$ points.
* **Space Complexity:** $O(N)$ — Storing all $N$ distances in memory.

### Optimized Approach (Max-Heap)
We combine the "VIP Club" and the "Negative Trick" templates!
1. **The Math Shortcut:** Computing square roots is slow. Since we only care about *comparing* distances, we can just use $x^2 + y^2$. If $A^2 < B^2$, then $A < B$.
2. **The VIP Club:** We want to keep a heap of size exactly `k` that holds the *smallest* distances. If we get $k + 1$ elements, we must pop the *largest* distance.
3. **The Negative Trick:** To make the largest distance pop out easily, we need a Max-Heap. We multiply our distance by `-1` before pushing it into Python's Min-Heap.
4. **The Heap Storage:** A heap can hold more than just numbers! We can push a Python "tuple" (a mini list) containing `(-distance, x, y)`. Python will automatically sort the heap using the first item in the tuple (the `-distance`).

* **Time Complexity:** $O(N \log k)$ — We process all $N$ points, but our heap operations only take $\log k$ time because the heap never grows larger than $k$.
* **Space Complexity:** $O(k)$ — We strictly cap the heap at size $k$, saving massive amounts of memory.

In [ ]:
import heapq
from typing import List

class Solution:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        max_heap = []
        
        for x, y in points:
            # 1. Calculate the distance (ignoring the square root for speed)
            # 2. Apply the Negative Trick so the BIGGEST distance acts as the "smallest" value
            dist = -1 * ((x ** 2) + (y ** 2))
            
            # 3. Push a tuple into the heap: (negative_distance, x-coord, y-coord)
            # Python will use the first item (dist) to decide who bubbles to the top
            heapq.heappush(max_heap, (dist, x, y))
            
            # 4. VIP Club Rule: If the heap exceeds size k, kick out the top element.
            # Because of the Negative Trick, the top element is the FARTHEST point!
            if len(max_heap) > k:
                heapq.heappop(max_heap)
                
        # 5. Compile the final list of surviving VIPs
        result = []
        for dist, x, y in max_heap:
            result.append([x, y])
            
        return result

### Problem 004: Kth Largest Element in an Array (LeetCode 215)

### Problem Definition and Constraints
Given an unsorted array of integers `nums` and an integer `k`, return the `k`-th largest element in the array. 
*Note: It is the k-th largest in sorted order, not the k-th distinct element (duplicates count).*

* Constraints:
  * 1 <= k <= nums.length <= 10^5
  * -10^4 <= nums[i] <= 10^4

### Brute Force Approach
Just use Python's built-in sort function to sort the array in descending order, then return the element at index `k - 1`. 
* **Time Complexity:** $O(N \log N)$ — Sorting the entire array is slow.
* **Space Complexity:** $O(1)$ or $O(N)$ — Depending on whether the sorting is done in-place or creates a new array.

### Optimized Approach (Min-Heap)
We use the exact same **"VIP Club"** template. We iterate through the array, pushing every number into a Min-Heap. The moment the heap size exceeds `k`, we pop the top element (which is the smallest number currently in the heap). 
By the time the loop finishes, the heap will hold exactly the `k` largest numbers from the array. Because it is a Min-Heap, the smallest of those `k` numbers (which is the actual `k`-th largest overall) will be sitting perfectly at the top (`min_heap[0]`).

* **Time Complexity:** $O(N \log k)$ — We process all $N$ elements, but pushing/popping only takes $\log k$ time because the heap size is strictly capped at $k$.
* **Space Complexity:** $O(k)$ — Our heap only ever stores exactly $k$ elements, discarding the rest.

In [ ]:
import heapq
from typing import List

class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        min_heap = []
        
        for num in nums:
            # 1. Let the new person into the club
            heapq.heappush(min_heap, num)
            
            # 2. If the club exceeds capacity, kick out the "poorest" member
            if len(min_heap) > k:
                heapq.heappop(min_heap)
                
        # 3. The poorest member of the surviving VIP club is the k-th largest element!
        return min_heap[0]

### Problem 005: Task Scheduler (LeetCode 621)

### Problem Definition and Constraints
You are given an array of CPU `tasks` (represented by letters A-Z) and a cooldown period `n`. 
* Each task takes exactly 1 cycle to run.
* You can run tasks in any order, but if you run a task (like 'A'), you cannot run another 'A' until exactly `n` cycles have passed. 
* During cooldowns, you can run other tasks. If no other tasks are available, the CPU sits "Idle".
Return the minimum total cycles needed to finish all tasks.

* Constraints:
  * 1 <= tasks.length <= 10^4
  * 0 <= n <= 100

### Brute Force Approach
Generate every single possible valid ordering of the tasks and simulate them to see which one takes the least amount of time. 
* **Time Complexity:** Exponential $O(K!)$ — Testing every permutation of tasks is incredibly slow and will result in a Time Limit Exceeded (TLE) error.
* **Space Complexity:** $O(K)$ — For the recursion call stack.

### Optimized Approach (Max-Heap + Cooldown Queue)
We use a **Greedy strategy**: always process the most frequent available task first. 
1. **Count Frequencies:** Count how many times each task appears. We don't actually care *which* letter it is, only its frequency.
2. **The Max-Heap:** Push all the frequencies into a Max-Heap (using the Negative Trick). This ensures the task with the highest remaining frequency is always at the top.
3. **The Cooldown Queue:** When a task runs, we subtract 1 from its frequency. If it still has tasks left to run, it goes into a `queue` (the cooldown room) along with the exact exact `time` it is allowed to come out.
4. **The Simulation:** A global `time` counter ticks up by 1 every loop. We pop from the Max-Heap to run a task. Then, we check the Queue. If the guy at the front of the Queue is done cooling down, we pop him out and push him back into the Max-Heap!

* **Time Complexity:** $O(M)$ — Where $M$ is the total number of tasks. Pushing and popping from the heap technically takes $O(\log 26)$, but since 26 (the alphabet) is a constant limit, it simplifies to $O(1)$ time per task!
* **Space Complexity:** $O(1)$ — The heap and the queue will never hold more than 26 items (the letters A-Z), making the space requirement constant regardless of how massive the input array is.

In [ ]:
import heapq
import collections
from typing import List

class Solution:
    def leastInterval(self, tasks: List[str], n: int) -> int:
        # 1. Count the frequencies of each task
        # Counter(['A','A','B']) -> {'A': 2, 'B': 1}
        count = collections.Counter(tasks)
        
        # 2. Build the Max-Heap using the Negative Trick
        # We only care about the counts, not the letters themselves
        max_heap = [-c for c in count.values()]
        heapq.heapify(max_heap)
        
        # 3. The Cooldown Room: will store tuples of (remaining_count, available_time)
        queue = collections.deque()
        
        # The master clock
        time = 0
        
        # 4. Run the CPU simulation
        # The CPU keeps running as long as there is stuff in the heap OR the waiting room
        while max_heap or queue:
            time += 1
            
            # If there is a task ready to run, do it!
            if max_heap:
                # Pop the most frequent task and simulate running it (add 1 to reduce the negative value)
                current_count = heapq.heappop(max_heap) + 1
                
                # If the task still has runs left, send it to the cooldown room!
                if current_count != 0:
                    # It can leave the room at current `time` + `n`
                    queue.append((current_count, time + n))
            
            # Check the cooldown room: Is the person at the front ready to come out?
            if queue and queue[0][1] == time:
                ready_task_count, _ = queue.popleft()
                heapq.heappush(max_heap, ready_task_count)
                
        return time